In [1]:
import sys
sys.path.append("..")

In [2]:
from src.data import load_data, filter_pair, list_pairs, get_one_hot
from src.features import load_descriptors, build_features, var_scaler, prior_yields
from src.optimize import expected_improvement, run_bo, run_random_search
from src.model import predict, fit_gp, fit_rf_baseline
from baseline.plots import plot_feature_importance, plot_trajectory, plot_tsne
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

2026-10-04 19:46:13.674 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-04 19:46:13.677 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [21]:
path_data = '../data/suzuki.xlsx'
path_desc = '../data/descriptors.json'
df = load_data(path_data)
descriptors = load_descriptors(path_desc)
df_desc = build_features(df, descriptors)
df_desc = var_scaler(df_desc)
features = [
    "polarity_solvent", "dielectric_solvent", "bp_solvent", "viscosity_solvent",
    "mw_reagent", "pKa_reagent", "solubility_h2o", "mw_ligand", "logP_ligand"
]
df_desc = df_desc[features]
reactant_pairs = list_pairs(df)


In [12]:
# max_iter does not include starting points
def benchmark(df, n_seeds=10, n_init=10, max_iter=30, rs_runs=100, one_hot = False, prior_values=False):
    rows = []
    for (r1, r2), filtered in df.groupby(['reactant1', 'reactant2']):
        one_hot_block = filtered[["ligand", "solvent", "reagent"]].copy()
        one_hot_block = get_one_hot(one_hot_block)
        prior_block = prior_yields(df, filtered).loc[filtered.index]
        descriptor_block = df_desc.loc[filtered.index]
        if one_hot:
            X = one_hot_block
        else:
            X = descriptor_block
        
        if prior_values:
            X = X.merge(prior_block, on="Reaction_No")
        X.index = filtered.index
        y = filtered["yield_uv"]
        if len(y) < max_iter + n_init:
            #skipping pairs with no experiments
            continue
        for seed in range(n_seeds):
            bo, max_start = run_bo(X, y, max_iter, n_init=n_init, init_seed=seed, prior_values=prior_values)
            rs = run_random_search(y, max_iter, rs_runs, n_init, init_seed=seed, seed=seed).mean(axis=0)
            max_yield = filtered["yield_uv"].max()
            for iter in range(max_iter):
                rows.append(dict(r1=r1, r2=r2, iter=iter, seed=seed, bo=bo[iter], rs=rs[iter], max_start = max_start, max_yield=max_yield))
    result = pd.DataFrame(rows)
    return result
            



In [13]:
def get_metrics(res: pd.DataFrame, max_iter: int, models: list) -> tuple:
    top_5 = {name: np.array([]) for name in models}
    reach_95 = {name: np.array([]) for name in models}
    for (r1, r2), pair_res in res.groupby(["r1", "r2"]):
        pair_res = res[(res["r1"]==r1) & (res["r2"]==r2)]
        thr = sorted(filter_pair(df, (r1, r2))["yield_uv"], reverse=True)[4]
        final_iter = pair_res[pair_res["iter"]==max_iter]
        for model in models:
            top_5[model] = np.append(top_5[model], np.count_nonzero(final_iter[model]>=thr)/10)
            reach_95[model] = np.append(reach_95[model], np.count_nonzero(final_iter[model]/final_iter["max_yield"] >= 0.95)/10)
    for model in models:
        top_5[model] = top_5[model].mean()
        reach_95[model] = reach_95[model].mean()
    return (top_5, reach_95)

In [14]:
res = pd.read_csv("../data/total_benchmark", sep=";", index_col="Unnamed: 0")

In [15]:
get_metrics(res, 30, ["rs"])

({'rs': np.float64(0.2)}, {'rs': np.float64(0.48000000000000004)})

In [16]:
filtered = filter_pair(df, ("6-chloroquinoline", "2a, Boronic Acid"))
y = filtered["yield_uv"].copy()
X = filtered.drop("yield_uv", axis=1)

In [22]:
benchmark(df, n_seeds=1, n_init=10, max_iter=30, rs_runs=1, one_hot=True, prior_values=True).to_csv("../data/test", sep=";")

In [20]:
benchmark(df.loc[filtered.index], n_seeds=1, n_init=10, max_iter=30, rs_runs=1, one_hot=True, prior_values=False)

,r1,r2,iter,seed,bo,rs,max_start,max_yield
0,6-chloroquinoline,"2a, Boronic Acid",0,0,88.925291,88.925291,88.925291,93.674515
1,6-chloroquinoline,"2a, Boronic Acid",1,0,88.966679,88.925291,88.925291,93.674515
2,6-chloroquinoline,"2a, Boronic Acid",2,0,88.966679,88.925291,88.925291,93.674515
3,6-chloroquinoline,"2a, Boronic Acid",3,0,88.966679,88.925291,88.925291,93.674515
4,6-chloroquinoline,"2a, Boronic Acid",4,0,88.966679,88.925291,88.925291,93.674515
5,6-chloroquinoline,"2a, Boronic Acid",5,0,88.966679,88.925291,88.925291,93.674515
6,6-chloroquinoline,"2a, Boronic Acid",6,0,88.966679,88.925291,88.925291,93.674515
7,6-chloroquinoline,"2a, Boronic Acid",7,0,93.674515,88.925291,88.925291,93.674515
8,6-chloroquinoline,"2a, Boronic Acid",8,0,93.674515,88.925291,88.925291,93.674515
9,6-chloroquinoline,"2a, Boronic Acid",9,0,93.674515,88.925291,88.925291,93.674515


In [24]:
myarr = run_random_search(y, 30, runs=100, n_init=10, init_seed=1, seed=0)

In [46]:
sum(myarr[:, -1]>90)

np.int64(38)

In [42]:
myarr[:, -1]

array([83.61894524, 92.22423865, 88.27536857, 92.83440729, 88.96667903,
       88.27536857, 92.83440729, 93.67451454, 88.96667903, 93.67451454,
       83.61894524, 85.94743124, 83.61894524, 90.30175352, 83.61894524,
       92.83440729, 89.16720838, 92.83440729, 87.82007096, 90.30175352,
       93.67451454, 90.30175352, 92.22423865, 91.34851062, 88.92529128,
       93.67451454, 83.61894524, 85.94743124, 92.22423865, 83.61894524,
       90.30175352, 87.82007096, 91.34851062, 91.34851062, 89.16720838,
       88.92529128, 91.34851062, 89.16720838, 93.67451454, 83.61894524,
       83.61894524, 88.96667903, 85.94743124, 83.61894524, 83.61894524,
       92.83440729, 93.67451454, 93.67451454, 92.83440729, 93.67451454,
       83.61894524, 85.21246875, 83.61894524, 83.61894524, 88.92529128,
       91.34851062, 92.22423865, 83.61894524, 92.83440729, 83.61894524,
       83.61894524, 89.16720838, 85.77342262, 89.16720838, 83.61894524,
       93.67451454, 85.94743124, 83.61894524, 88.92529128, 89.16